#  Credit Risk – Silver Layer

## Ziel

Im Silver Layer werden die Rohdaten aus dem Bronze Layer bereinigt und für die weitere Analyse vorbereitet.

Die Daten werden aus der Tabelle `Data_Science.credit_risk_bronze` gelesen.

## Verarbeitungsschritte

Nach dem Laden der Bronze-Tabelle wird die Silver-Schicht schrittweise aufgebaut.

1. **Bronze-Daten laden**
2. **Fehlende Werte standardisieren**
   - `NA`, `N/A` und leere Zeichenketten werden in `NULL` umgewandelt
3. **Datenqualität und Formate validieren**
4. **Datentypen und Datenformate prüfen und korrigieren**
5. **Variablen mit hohem Missing-Anteil identifizieren und entfernen**
6. **Duplikate prüfen und entfernen**
7. **Datenqualität der bereinigten Daten validieren**
8. **Bereinigte Daten als Silver-Tabelle speichern**


```text
1. Bronze-Tabelle laden
        ↓
2. NaN / "NA" → NULL
        ↓
3. Validierung der Daten
        ↓
4. Datenformate und Datentypen bereinigen
        ↓
5. Missing-Value-Analyse
        ↓
6. Duplikate prüfen und entfernen
        ↓
7. Datenqualität validieren
        ↓
8. Silver-Tabelle speichern

In [0]:
### 2. Bronze-Daten laden

# Bronze-Tabelle laden
df_silver = spark.table("Data_Science.credit_risk_bronze")

display(df_silver.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65%,162.87,B,B2,null,10+ years,RENT,24000,Verified,Dec-2011,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,735.0,739.0,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,Jan-2015,171.62,Dec-2018,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


**Schema prüfen**

In [0]:
df_silver.printSchema()

print("Anzahl Zeilen:", df_silver.count())
print("Anzahl Spalten:", len(df_silver.columns))

root
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: integer (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: 

## 2. Fehlende Werte standardisieren

Nach dem Laden der Bronze-Tabelle werden unterschiedliche Darstellungen von
fehlenden Werten vereinheitlicht.

Im Rohdatensatz können fehlende Werte beispielsweise als `NA`, `N/A` oder als
leere Zeichenketten gespeichert sein.

Diese Werte werden in einheitliche `NULL`-Werte umgewandelt.

Dadurch können fehlende Werte in den nachfolgenden Verarbeitungsschritten
konsistent erkannt und behandelt werden.

### Transformation

- `NA` → `NULL`
- `N/A` → `NULL`
- leere Zeichenkette → `NULL`

Die Standardisierung stellt ausschließlich die technische Konsistenz der
Daten sicher. Es erfolgt an dieser Stelle keine Ersetzung der fehlenden Werte
durch beispielsweise `0`, Mittelwert oder Median.

In [0]:
from pyspark.sql.functions import col, trim, when

# Definition der Darstellungen für fehlende Werte
missing_values = ["NA", "N/A", ""]

# Fehlende Werte in allen Spalten standardisieren
for c in df_silver.columns:
    df_silver = df_silver.withColumn(
        c,
        when(
            trim(col(c).cast("string")).isin(missing_values),
            None
        ).otherwise(col(c))
    )

## Validierung der Standardisierung

Nach der Transformation wird überprüft, ob die definierten Darstellungen
fehlender Werte erfolgreich in `NULL` umgewandelt wurden.

In [0]:
# Anzahl der verbleibenden "NA"- und "N/A"-Werte prüfen

for value in ["NA", "N/A", ""]:
    print(f"Wert '{value}':")
    
    for c in df_silver.columns:
        count = df_silver.filter(
            trim(col(c).cast("string")) == value
        ).count()
        
        if count > 0:
            print(f"  {c}: {count}")

Wert 'NA':
Wert 'N/A':
Wert '':


### Ergebnis

Die Prüfung zeigt, dass keine Werte mit den Darstellungen `NA`, `N/A` oder
leeren Zeichenketten mehr vorhanden sind.

Die entsprechenden fehlenden Werte wurden erfolgreich in `NULL` standardisiert.

Damit ist die Vereinheitlichung der fehlenden Werte abgeschlossen.

## Validierung der Spalte `int_rate` und `revol_util`

Bevor Datentypen und Datenformate angepasst werden, wird die Spalte `int_rate` zunächst auf ihre ursprüngliche Struktur und Datenqualität geprüft.

Die Spalte enthält Zinssätze, die im Rohdatensatz als `STRING` gespeichert sind. Die Werte enthalten dabei das Prozentzeichen `%`, zum Beispiel:

- `10.65%`
- `13.99%`
- `8.49%`

Eine direkte Konvertierung von `STRING` zu `DOUBLE` ist aufgrund des Prozentzeichens nicht möglich.

Daher wird vor der eigentlichen Transformation überprüft:

- welcher Datentyp aktuell vorliegt
- welche Werte in der Spalte enthalten sind
- ob fehlende Werte vorhanden sind
- ob alle Werte dem erwarteten Prozentformat entsprechen

Die Validierung erfolgt bewusst **vor der Bereinigung und Datentypkonvertierung**, damit der ursprüngliche Zustand der Daten nachvollziehbar dokumentiert ist.

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace

df_silver = df_silver.withColumn(
    "int_rate",
    regexp_replace(
        trim(col("int_rate")),
        "%",
        ""
    ).cast("double")
)

df_silver = df_silver.withColumn(
    "revol_util",
    regexp_replace(
        trim(col("revol_util")),
        "%",
        ""
    ).cast("double")
)

##### Kleine Prüfung

In [0]:
df_silver.select(
    "int_rate",
    "revol_util"
).show(10)

df_silver.select(
    "int_rate",
    "revol_util"
).printSchema()

+--------+----------+
|int_rate|revol_util|
+--------+----------+
|   10.65|      83.7|
|   15.27|       9.4|
|   15.96|      98.5|
|   13.49|      21.0|
|   12.69|      53.9|
|     7.9|      28.3|
|   15.96|      85.6|
|   18.64|      87.5|
|   21.28|      32.6|
|   12.69|      36.5|
+--------+----------+
only showing top 10 rows
root
 |-- int_rate: double (nullable = true)
 |-- revol_util: double (nullable = true)



## 2. Datenformate und Datentypen bereinigen

Die Rohdaten enthalten verschiedene Datentypen. Einige numerische Variablen wurden beim Import als `string` erkannt.

Im Silver Layer werden diese Variablen in geeignete numerische Datentypen konvertiert.

Ziel ist eine konsistente Datenstruktur für die weitere Verarbeitung.

In [0]:
from pyspark.sql.functions import col

# Numerische Spalten
double_cols = [
     'last_fico_range_low', 'total_pymnt_inv','installment',  'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',  'total_acc', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'last_fico_range_high', 'collections_12_mths_ex_med', 'policy_code', 'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt', 'pub_rec_bankruptcies', 'tax_liens',"int_rate","revol_util"
]

from pyspark.sql.functions import expr
  ## Konvertierung von Double Attributen
for c in double_cols:
    if c in df_silver.columns:
        df_bronze = df_silver.withColumn(
            c,
            expr(f"try_cast(`{c}` AS DOUBLE)")
      )
        
    ## Konvertierung von Integer Attributen
    int_cols = ["id", "loan_amnt","funded_amnt","funded_amnt_inv","annual_inc"]
    for c in int_cols:
        if c in df_silver.columns:
           df_silver = df_bronze.withColumn(
              c,
               expr(f"try_cast(`{c}` AS INTEGER)")
       )

Schema prüfen

In [0]:
df_silver.printSchema()

root
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: integer (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: 

%md
## Prüfung der Spalten, die den Wert `major_purchase` enthalten

Zur Überprüfung der Datenstruktur wurde nach dem Wert `major_purchase` in den
String-Spalten gesucht.

Der Wert `major_purchase` gehört zur Spalte `purpose`, da diese Spalte den
Verwendungszweck des Kredits beschreibt.

Die Spalte `purpose` ist daher als **kategorische Variable (`STRING`)** zu
behandeln und darf nicht in eine numerische Variable (`DOUBLE` oder `BIGINT`)
konvertiert werden.

Die weiteren gefundenen String-Spalten werden nicht aufgrund dieses Suchwertes
als numerisch oder kategorisch klassifiziert. Die Datentypen werden anhand der
fachlichen Bedeutung und der Struktur des Datensatzes bestimmt.


In [0]:
from pyspark.sql.functions import col

for c in df_silver.columns:
    if df_silver.filter(
        col(c).cast("string") == "major_purchase"
    ).limit(1).count() > 0:
        print(f"Gefunden in Spalte: {c}")

Gefunden in Spalte: purpose
Gefunden in Spalte: title
Gefunden in Spalte: zip_code
Gefunden in Spalte: addr_state


## Validierung der kategorialen und textuellen Spalten

Die Überprüfung der betroffenen Spalten zeigt, dass `purpose`, `title`,
`zip_code`, `addr_state` und `earliest_cr_line` textuelle Informationen
enthalten.

Die Spalten werden daher nicht als numerische Variablen behandelt.

- `purpose`: enthält Kategorien für den Verwendungszweck des Kredits.
- `title`: enthält frei eingegebene Bezeichnungen des Kredits.
- `zip_code`: enthält anonymisierte Postleitzahlen im Format `xxx` bzw. `xxx`.
- `addr_state`: enthält die Abkürzung des Bundesstaates.
- `earliest_cr_line`: enthält Monats- und Jahresangaben im Format `Mon-YY`.

Die Untersuchung bestätigt, dass der Wert `major_purchase` ein kategorialer
Wert der Spalte `purpose` ist und nicht numerisch konvertiert werden darf.


## Festlegung der Datentypen

Für die überprüften fünf Spalten werden folgende Datentypen festgelegt:

| Spalte | Datentyp | Begründung |
|---|---|---|
| `purpose` | `STRING` | Kreditverwendungszweck |
| `title` | `STRING` | Textuelle Bezeichnung des Kredits |
| `zip_code` | `STRING` | Enthält anonymisierte Postleitzahlen wie `860xx` |
| `addr_state` | `STRING` | Abkürzung des Bundesstaates |
| `earliest_cr_line` | `DATE` | Datum des ältesten Kreditkontos |

Die Spalten `purpose`, `title`, `zip_code` und `addr_state` bleiben als
`STRING` erhalten.

Die Spalte `earliest_cr_line` wird anschließend vom Format `MMM-yy` in den
Datentyp `DATE` konvertiert.

In [0]:
df_silver.select(
    "purpose",
    "title",
    "zip_code",
    "addr_state",
    "earliest_cr_line",
    "fico_range_low"
).printSchema()



root
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- fico_range_low: double (nullable = true)



Für numerische Spalten verwenden wir try_cast. Dadurch werden nicht konvertierbare Werte automatisch zu NULL.



### Bereinigung ungültiger Werte

Der Wert `major_purchase` in der Spalte `earliest_cr_line` wird als ungültiger Wert identifiziert und durch `NULL` ersetzt.

In [0]:
from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "earliest_cr_line",
    when(col("earliest_cr_line") == "major_purchase", None)
    .otherwise(col("earliest_cr_line"))
)

##### Konvertierung der folgenden Attribute in das Datumsformat

In [0]:
from pyspark.sql.functions import to_date, col

date_cols = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "last_credit_pull_d"
]

for c in date_cols:
    df_silver = df_silver.withColumn(
        c,
        to_date(col(c), "MMM-yyyy")
    )

In [0]:
df_silver.select( "earliest_cr_line").groupBy( "earliest_cr_line").count().orderBy("count", ascending=False).show(1)

+----------------+-----+
|earliest_cr_line|count|
+----------------+-----+
|        Nov-1998|  368|
+----------------+-----+
only showing top 1 row


In [0]:
df_silver.select(
    "earliest_cr_line",
    "last_pymnt_d",
    "last_credit_pull_d",
    "issue_d"
).printSchema()

root
 |-- earliest_cr_line: date (nullable = true)
 |-- last_pymnt_d: date (nullable = true)
 |-- last_credit_pull_d: date (nullable = true)
 |-- issue_d: date (nullable = true)



## 3. Duplikate prüfen und  entfernen

Im nächsten Schritt prüfen wir den Datensatz auf doppelte Datensätze.

Duplikate können zu einer Verzerrung der späteren Analyse und Modellierung führen.

Doppelte Zeilen werden identifiziert und entfernt.

**Duplikate entfernen**

In [0]:
# Anzahl aller Zeilen
total_rows = df_silver.count()

# Anzahl eindeutiger Zeilen
distinct_rows = df_silver.distinct().count()

# Anzahl Duplikate
duplicate_rows = total_rows - distinct_rows

print("Gesamtzahl Zeilen:", total_rows)
print("Eindeutige Zeilen:", distinct_rows)
print("Duplikate:", duplicate_rows)

Gesamtzahl Zeilen: 39717
Eindeutige Zeilen: 39717
Duplikate: 0


In [0]:
display(df_silver.limit(1) )

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65%,162.87,B,B2,null,10+ years,RENT,24000,Verified,Dec-2011,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,735.0,739.0,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,Jan-2015,171.62,Dec-2018,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


In [0]:
from pyspark.sql.functions import col

double_cols = [
    field.name
    for field in df_silver.schema.fields
    if field.dataType.simpleString() == "double"
]

print("Anzahl DOUBLE-Spalten:", len(double_cols))
print(double_cols)

Anzahl DOUBLE-Spalten: 31
['int_rate', 'installment', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'policy_code', 'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt', 'pub_rec_bankruptcies', 'tax_liens']


%md
## 3. Silver-Tabelle speichern

In [0]:
bronze_credit_risk= "Data_Science.credit_risk_silver"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_credit_risk)
)

## 4. Kontrolle der Bronze-Tabelle

In [0]:
df_silver_check = spark.table("Data_Science.credit_risk_silver")

display(df_silver_check.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65,162.87,B,B2,null,10+ years,RENT,24000,Verified,2011-12-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,1985-01-01,735.0,739.0,1.0,3.0,0.0,13648.0,83.7,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,2015-01-01,171.62,2018-12-01,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


### Medallion Architecture bis jetzt

Die Datenpipeline ist aktuell bis zur **Silver-Schicht** aufgebaut. 
Dabei werden die Rohdaten zunächst aus der CSV-Datei eingelesen und in der 
Bronze-Schicht technisch verfügbar gemacht. Anschließend werden die Daten 
in der Silver-Schicht durch Data-Engineering-Schritte bereinigt und 
strukturiert.

```text


dataset.csv
    │
    ▼
Credit_Analytics_Risk_bronze
    │
    ▼
credit_risk_bronze
    │
    ▼
Credit_Analytics_Risk_silver
    │
    ▼
credit_risk_silver


````


### Struktur der Silver-Tabelle

Die Tabelle `credit_risk_silver` enthält die technisch aufbereiteten Daten aus der Bronze-Schicht. 
In der Silver-Schicht wurden insbesondere Datentypen vereinheitlicht und Datums- sowie numerische Attribute entsprechend ihrer vorgesehenen Verwendung konvertiert.

Die Silver-Tabelle enthält aktuell **63 Attribute**.

#### Datentypen

- **Numerische Attribute:** `integer` bzw. `double`
- **Datumsattribute:** `date`
- **Kategoriale und textuelle Attribute:** `string`

#### Datumsattribute

Folgende Attribute liegen als `date` vor:

- `issue_d`
- `earliest_cr_line`
- `last_pymnt_d`
- `last_credit_pull_d`

Ungültige bzw. nicht konvertierbare Datumswerte werden als `NULL` gespeichert.

#### Numerische Attribute

Numerische Attribute wie beispielsweise

- `loan_amnt`
- `funded_amnt`
- `int_rate`
- `annual_inc`
- `dti`
- `fico_range_low`
- `fico_range_high`
- `last_fico_range_low`
- `last_fico_range_high`
- `total_pymnt`
- `recoveries`

liegen als `integer` oder `double` vor.

Nicht numerisch interpretierbare Werte in den dafür vorgesehenen numerischen Attributen werden als `NULL` behandelt.

#### String-Attribute

Kategoriale und textuelle Attribute bleiben als `string` erhalten, beispielsweise:

- `term`
- `grade`
- `sub_grade`
- `emp_title`
- `home_ownership`
- `verification_status`
- `loan_status`
- `purpose`
- `title`
- `zip_code`
- `addr_state`
- `application_type`

### Ergebnis der Silver-Schicht

Die Tabelle `credit_risk_silver` stellt damit eine **technisch standardisierte und strukturierte Datengrundlage** für die nachfolgenden Verarbeitungsschritte bereit.

Eine fachliche Modellierung oder Spezialisierung für **PD oder LGD** erfolgt in dieser Schicht noch nicht.

## 6. Validierung der Silver-Tabelle

Nach der technischen Bereinigung wird die Silver-Tabelle abschließend auf
Datenqualität und Konsistenz geprüft.

Dabei werden folgende Punkte kontrolliert:

1. Anzahl der Zeilen und Spalten
2. Datentypen der Attribute
3. Fehlende Werte (`NULL`)
4. Duplikate
5. Gültigkeit der Datumsattribute
6. Gültigkeit der numerischen Attribute
7. Plausibilität der wichtigsten Wertebereiche

Die Validierung dient dazu sicherzustellen, dass die Silver-Tabelle technisch
konsistent ist und als Grundlage für die nächste Verarbeitungsschicht
verwendet werden kann.

**1. Anzahl der Zeilen & Spalten und Prüfung von Datentypen der Attribute**

In [0]:
# Anzahl Zeilen und Spalten
print("Zeilen:", df_silver_check.count())
print("Spalten:", len(df_silver_check.columns))

# Schema
df_silver_check.printSchema()

# Duplikate
total_rows = df_silver_check.count()
distinct_rows = df_silver_check.distinct().count()

print("Gesamtzeilen:", total_rows)
print("Eindeutige Zeilen:", distinct_rows)
print("Duplikate:", total_rows - distinct_rows)

Zeilen: 39717
Spalten: 60
root
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: integer (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: date (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = 

**3. Fehlende Werte (NULL)**

Zunächst wird überprüft, wie viele fehlende Werte in den einzelnen Attributen
der Silver-Tabelle vorhanden sind.

Die Prüfung dient der abschließenden Kontrolle der vorherigen Bereinigung.

In [0]:
from pyspark.sql.functions import col, sum, when, lit

total_rows = df_silver.count()

# NULL-Anzahl je Attribut
missing_counts = df_silver.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
])

# In Dictionary umwandeln
missing_dict = missing_counts.collect()[0].asDict()

# Nur Attribute mit NULL-Werten
result = [
    (c, int(count))
    for c, count in missing_dict.items()
    if count > 0
]

# Spark DataFrame erstellen
df_missing = spark.createDataFrame(
    result,
    ["Attribut", "NULL_Anzahl"]
)

# NULL-Anteil anschließend mit Spark berechnen
df_missing = df_missing.withColumn(
    "NULL_Anteil_%",
    col("NULL_Anzahl") / lit(total_rows) * 100
)

display(
    df_missing.orderBy(
        col("NULL_Anteil_%").desc()
    )
)

Attribut,NULL_Anzahl,NULL_Anteil_%
desc,13152,33.114283556159826
emp_title,2459,6.191303472064859
emp_length,1075,2.7066495455346575
pub_rec_bankruptcies,725,1.8254148097791876
last_pymnt_d,294,0.7402371780345948
dti,225,0.5665080444142306
earliest_cr_line,225,0.5665080444142306
last_credit_pull_d,187,0.4708311302464939
delinq_2yrs,176,0.44313518140846486
fico_range_low,150,0.3776720296094871


**4. Duplikate**

Im nächsten Schritt wird geprüft, ob die Silver-Tabelle doppelte Datensätze
enthält.

Dabei werden vollständig identische Zeilen als Duplikate betrachtet.

In [0]:
# Anzahl aller Zeilen
total_rows = df_silver.count()

# Anzahl eindeutiger Zeilen
distinct_rows = df_silver.distinct().count()

# Anzahl der Duplikate
duplicate_rows = total_rows - distinct_rows

print("Gesamtzahl Zeilen:", total_rows)
print("Eindeutige Zeilen:", distinct_rows)
print("Duplikate:", duplicate_rows)

Gesamtzahl Zeilen: 39717
Eindeutige Zeilen: 39717
Duplikate: 0


**5. Gültigkeit der Datumsattribute**

Die Datumsattribute werden abschließend auf ihren Datentyp und auf nicht
konvertierbare bzw. fehlende Werte geprüft.

Folgende Attribute werden als `DATE` erwartet:

- `issue_d`
- `earliest_cr_line`
- `last_pymnt_d`
- `last_credit_pull_d`

Nicht gültige Datumswerte wurden bei der Konvertierung als `NULL` gespeichert.

In [0]:
date_cols = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "last_credit_pull_d"
]

for c in date_cols:
    print(
        f"{c}:",
        df_silver_check.schema[c].dataType.simpleString()
    )

issue_d: date
earliest_cr_line: date
last_pymnt_d: date
last_credit_pull_d: date


**NULL-Werte in den Datumsattributen prüfen**

In [0]:
from pyspark.sql.functions import col, sum, when

date_nulls = df_silver.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in date_cols
])

display(date_nulls)

issue_d,earliest_cr_line,last_pymnt_d,last_credit_pull_d
0,225,294,187


**6. Gültigkeit der numerischen Attribute**

Die numerischen Attribute werden abschließend auf ihren Datentyp und auf
fehlende Werte geprüft.

Nicht numerisch interpretierbare Werte wurden bei der Konvertierung durch
`NULL` ersetzt.

Dadurch wird sichergestellt, dass die als numerisch definierten Attribute
keine ungültigen String-Werte mehr enthalten.

In [0]:
numeric_cols_existing = [
    c for c in numeric_cols
    if c in df_silver.columns
]

for c in numeric_cols_existing:
    print(
        f"{c}:",
        df_silver_check.schema[c].dataType.simpleString()
    )

id: int
loan_amnt: int
funded_amnt: int
last_fico_range_low: double
funded_amnt_inv: int
total_pymnt_inv: double
int_rate: double
installment: double
annual_inc: int
dti: double
delinq_2yrs: double
fico_range_low: double
fico_range_high: double
inq_last_6mths: double
open_acc: double
pub_rec: double
revol_bal: double
revol_util: double
total_acc: double
out_prncp: double
out_prncp_inv: double
total_pymnt: double
total_rec_prncp: double
total_rec_int: double
total_rec_late_fee: double
recoveries: double
collection_recovery_fee: double
last_pymnt_amnt: double
last_fico_range_high: double
collections_12_mths_ex_med: double
policy_code: double
acc_now_delinq: double
chargeoff_within_12_mths: double
delinq_amnt: double
pub_rec_bankruptcies: double
tax_liens: double


**NULL-Werte prüfen**

In [0]:
from pyspark.sql.functions import col, sum, when

numeric_nulls = df_silver.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in numeric_cols_existing
])

display(numeric_nulls)

id,loan_amnt,funded_amnt,last_fico_range_low,funded_amnt_inv,total_pymnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_amnt,last_fico_range_high,collections_12_mths_ex_med,policy_code,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,0,0,66,0,35,0,0,0,225,176,150,122,101,79,64,47,87,36,69,62,48,22,14,20,17,14,60,84,103,36,71,118,52,725,61


**7. Plausibilität der wichtigsten Wertebereiche**

Abschließend werden ausgewählte numerische Attribute auf plausible
Wertebereiche geprüft.

Die Prüfung dient dazu, offensichtlich fehlerhafte oder unplausible Werte
zu identifizieren.

Dabei werden die Daten nicht verändert. Auffällige Werte werden lediglich
für eine mögliche weitere Prüfung ausgewiesen.

In [0]:
from pyspark.sql.functions import col

# Plausibilitätsprüfungen
checks = {
    "loan_amnt < 0": df_silver.filter(col("loan_amnt") < 0).count(),
    "funded_amnt < 0": df_silver.filter(col("funded_amnt") < 0).count(),
    "annual_inc < 0": df_silver.filter(col("annual_inc") < 0).count(),
    "int_rate < 0": df_silver.filter(col("int_rate") < 0).count(),
    "dti < 0": df_silver.filter(col("dti") < 0).count(),
    "fico_range_low < 0": df_silver.filter(col("fico_range_low") < 0).count(),
    "fico_range_high < 0": df_silver.filter(col("fico_range_high") < 0).count(),
    "revol_util < 0": df_silver.filter(col("revol_util") < 0).count(),
    "open_acc < 0": df_silver.filter(col("open_acc") < 0).count(),
    "total_acc < 0": df_silver.filter(col("total_acc") < 0).count()
}

for check, count in checks.items():
    print(f"{check}: {count}")

loan_amnt < 0: 0
funded_amnt < 0: 0
annual_inc < 0: 0
int_rate < 0: 0
dti < 0: 0
fico_range_low < 0: 0
fico_range_high < 0: 0
revol_util < 0: 0
open_acc < 0: 0
total_acc < 0: 0
